# 위험지대 객체 감지 모델 학습 (YOLO11n → ONNX)

Roboflow에서 받은 **SafeWalkBD v9** 에 **계단 데이터셋 2개**(katti/Stairs 516장, stairs-blind/Stairs_Detection 602장)를 합쳐
앱용 5개 클래스(crosswalk, stairs, pole, person, vehicle)로 정리하고, YOLO11n을 학습한 뒤 브라우저(ONNX Runtime Web)용 `best.onnx`를 만드는 노트북입니다.

**실행 전 확인**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택
2. Google Drive "내 드라이브" 바로 아래에 zip 3개가 있어야 합니다: `SafeWalkBD.v9i.yolov8.zip`, `Stairs.v1i.yolov8.zip`, `Stairs_Detection.v2i.yolov8.zip`
3. 2번 셀에서 `tools/merge_yolo_datasets.py` (저장소 최신 버전) 를 업로드

셀을 위에서부터 순서대로 실행하면 됩니다. (3-B 셀은 건너뜀)


In [ ]:
# 1. 패키지 설치
!pip install -q ultralytics roboflow
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# 2. 병합 스크립트 준비 (저장소의 tools/merge_yolo_datasets.py 를 업로드)
import os
if not os.path.exists("merge_yolo_datasets.py"):
    from google.colab import files
    print("tools/merge_yolo_datasets.py 파일을 선택해 업로드하세요.")
    up = files.upload()
    assert "merge_yolo_datasets.py" in up, "merge_yolo_datasets.py 가 업로드되지 않았습니다"
print("OK")

## 3. 데이터셋 가져오기 — 둘 중 하나만 실행

- **3-A**: PC에 받아 둔 `SafeWalkBD.v9i.yolov8.zip` 을 Google Drive 에 올린 뒤 푸는 방법 (API 키 불필요, 권장)
- **3-B**: Roboflow SDK 로 직접 내려받는 방법 (Private API Key 필요, zip 업로드 불필요)

In [ ]:
# 3-A. Google Drive 에 올린 zip 풀기
#   Drive 의 "내 드라이브" 바로 아래에 zip 을 올린 뒤 파일 이름이 다르면 아래를 고치세요.
from google.colab import drive
drive.mount("/content/drive")

ZIPS = {
    "safewalkbd": "/content/drive/MyDrive/SafeWalkBD.v9i.yolov8.zip",
    "stairs_katti": "/content/drive/MyDrive/Stairs.v1i.yolov8.zip",
    "stairs_blind": "/content/drive/MyDrive/Stairs_Detection.v2i.yolov8.zip",
}
import os, zipfile
for folder, zpath in ZIPS.items():
    assert os.path.exists(zpath), f"파일 없음: {zpath}  (Drive 에서 실제 파일 이름을 확인하세요)"
    os.makedirs(f"datasets/{folder}", exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(f"datasets/{folder}")
    print("풀기 완료:", folder, "->", os.listdir(f"datasets/{folder}"))
!cat datasets/*/data.yaml


In [ ]:
# 3-B. Roboflow SDK 로 포크한 데이터셋 다운로드 (3-A 를 했다면 건너뛰기)
#    WORKSPACE / 프로젝트 슬러그는 Roboflow 프로젝트 URL 에서 확인:
#    https://app.roboflow.com/<WORKSPACE>/<PROJECT_SLUG>/...
from getpass import getpass
from roboflow import Roboflow

WORKSPACE = "여기에-내-워크스페이스-슬러그"
PROJECTS = {
    # 로컬폴더이름: (프로젝트 슬러그, 버전 번호) — 포크한 프로젝트의 URL 기준
    "safewalkbd": ("safewalkbd-슬러그", 1),
}

rf = Roboflow(api_key=getpass("Roboflow Private API Key: "))
for folder, (slug, ver) in PROJECTS.items():
    print(f"\n== {folder}: {WORKSPACE}/{slug} v{ver}")
    rf.workspace(WORKSPACE).project(slug).version(ver).download("yolov8", location=f"datasets/{folder}")
!ls datasets/*
!cat datasets/*/data.yaml

In [ ]:
# 4-a. 클래스 매핑 확인 (dry-run: 파일 복사 없음)
#   SafeWalkBD v9: Crosswalk→crosswalk, Stairs→stairs, Pole→pole, Person→person, Train/Vehicle→vehicle, 나머지 제외
#   계단 데이터셋: Stairs/downstair/upstair → stairs,  Stairs_Detection 의 '0','1','2' 는 사진으로 확인한 결과 모두 계단 → stairs
EXTRA_MAP = ["0=stairs", "1=stairs", "2=stairs"]
SRCS = "--src datasets/safewalkbd --src datasets/stairs_katti --src datasets/stairs_blind"
map_args = " ".join(f'--map "{m}"' for m in EXTRA_MAP)
!python merge_yolo_datasets.py {SRCS} --out datasets/merged --dry-run {map_args}


In [ ]:
# 4-b. 실제 병합
!rm -rf datasets/merged
!python merge_yolo_datasets.py {SRCS} --out datasets/merged {map_args}
!cat datasets/merged/data.yaml


In [ ]:
# 5. 학습 (train 약 8,000장 × 60 epoch, T4 기준 약 60~80분)
#    - imgsz=320 : 스마트폰 속도 우선
#    - 중간에 끊기면 이 셀을 다시 실행 (resume=True 로 이어서 학습)
from ultralytics import YOLO
import glob, os, shutil

last = sorted(glob.glob("**/hazard2/weights/last.pt", recursive=True), key=os.path.getmtime)
if last:
    print("이어서 학습:", last[-1])
    model = YOLO(last[-1]); results = model.train(resume=True)
else:
    model = YOLO("yolo11n.pt")
    results = model.train(data="datasets/merged/data.yaml", epochs=60, imgsz=320, batch=32, patience=15,
                          project="runs", name="hazard2", exist_ok=True)

# 런타임이 초기화돼도 잃지 않도록 best.pt 를 Drive 에 백업
best = sorted(glob.glob("**/hazard2/weights/best.pt", recursive=True), key=os.path.getmtime)[-1]
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy(best, "/content/drive/MyDrive/hazard2_best.pt")
    print("Drive 백업 완료: /content/drive/MyDrive/hazard2_best.pt")


In [ ]:
# 6. 학습 결과 확인
#    Ultralytics 버전에 따라 결과 폴더가 runs/hazard 또는 runs/detect/runs/hazard 에 생기므로 자동 탐색
#    (런타임이 초기화된 경우를 대비해 설치부터 다시 확인)
!pip install -q ultralytics
import glob, os, shutil
from IPython.display import Image, display
from ultralytics import YOLO

cands = sorted(glob.glob("**/hazard2/weights/best.pt", recursive=True), key=os.path.getmtime)
if not cands and os.path.exists("/content/drive/MyDrive/hazard2_best.pt"):
    os.makedirs("runs/hazard2/weights", exist_ok=True)
    shutil.copy("/content/drive/MyDrive/hazard2_best.pt", "runs/hazard2/weights/best.pt")
    cands = ["runs/hazard2/weights/best.pt"]
    print("Drive 백업본을 복원했습니다.")
assert cands, "best.pt 를 찾지 못했습니다. 5번 셀(학습)을 다시 실행하세요."
RUN_DIR = os.path.dirname(os.path.dirname(cands[-1]))
print("결과 폴더:", RUN_DIR)
for png in ["results.png", "confusion_matrix_normalized.png"]:
    p = os.path.join(RUN_DIR, png)
    if os.path.exists(p):
        display(Image(p, width=900 if png == "results.png" else 600))

m = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))
metrics = m.val(data="datasets/merged/data.yaml", imgsz=320, plots=False)
print("mAP50:", round(float(metrics.box.map50), 3), " mAP50-95:", round(float(metrics.box.map), 3))
for i, name in m.names.items():
    print(f"  {name:<10} AP50={metrics.box.ap50[i]:.3f}")

In [ ]:
# 7. ONNX 내보내기 (브라우저용)
#    - opset=12 : WebGPU 호환
#    - nms=True : NMS 를 모델 안에 포함 → 출력 [1, 300, 6] (x1,y1,x2,y2,conf,cls)
#      (이 옵션이 지원되지 않는 버전이면 nms 없이 내보내고 JS 쪽에서 NMS 처리)
import json, shutil, glob, os
from ultralytics import YOLO

cands = sorted(glob.glob("**/hazard2/weights/best.pt", recursive=True), key=os.path.getmtime)
if not cands and os.path.exists("/content/drive/MyDrive/hazard2_best.pt"):
    cands = ["/content/drive/MyDrive/hazard2_best.pt"]
assert cands, "best.pt 를 찾지 못했습니다."
m = YOLO(cands[-1])
try:
    onnx_path = m.export(format="onnx", opset=12, imgsz=320, nms=True, simplify=True)
    nms_in_model = True
except Exception as e:
    print("nms=True 내보내기 실패, NMS 없이 재시도:", e)
    onnx_path = m.export(format="onnx", opset=12, imgsz=320, simplify=True)
    nms_in_model = False

shutil.copy(onnx_path, "best.onnx")
labels = {
    "names": [m.names[i] for i in range(len(m.names))],
    "imgsz": 320,
    "nms_in_model": nms_in_model,
}
json.dump(labels, open("labels.json", "w"), ensure_ascii=False, indent=2)
print(open("labels.json").read())
!ls -lh best.onnx

In [ ]:
# 8. ONNX 모델 동작 확인 (검증 이미지 1장)
import glob
from ultralytics import YOLO
from IPython.display import Image, display

sample = sorted(glob.glob("datasets/merged/valid/images/*"))[0]
onnx_model = YOLO("best.onnx", task="detect")
r = onnx_model.predict(sample, imgsz=320, conf=0.3)[0]
r.save("onnx_check.jpg")
print("감지:", [(onnx_model.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])
display(Image("onnx_check.jpg", width=500))

In [ ]:
# 9. 결과 파일 내려받기 → 저장소 루트에 best.onnx, labels.json 으로 넣기
from google.colab import files
files.download("best.onnx")
files.download("labels.json")

## 다음 단계

1. 내려받은 `best.onnx`, `labels.json` 을 저장소 루트에 둔다.
2. 웹앱(`main.js`)이 ONNX Runtime Web 으로 교체되어 있으면 파일만 바꿔도 바로 동작한다.
3. 정확도가 아쉬우면: 직접 찍은 사진을 Roboflow 에서 라벨링해 데이터셋에 추가 → 3번 셀부터 다시 실행.